In [37]:
from pydantic import BaseModel,Field # returns Dictionary
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate # PromptTemplate --> Directly Prompt not like ChatPromptTemplate()
from langchain_core.output_parsers import PydanticOutputParser
import os

In [38]:
'''
Why Pydantic,
For structring the output.
In real-time --> To validate the response or request
'''

'\nWhy Pydantic,\nFor structring the output.\nIn real-time --> To validate the response or request\n'

In [39]:
with open('data.txt','r+') as file:
    data = file.read()

In [40]:
qwen_model = ChatOllama(
    model = 'qwen2.5:1.5b',
    temperature=0
)
gemini_model = ChatGoogleGenerativeAI(
    model = 'gemini-3.6-flash',
    api_key = os.environ['Gemini_api'],
    temperature = 0
)

In [41]:
template = ''' 
You are a helpful assitant that can answer questions about the product.
Answer the question based on the product details provided.
product : {product}
question : {question}
'''

In [42]:
prompt_template = PromptTemplate.from_template(template)

In [43]:
# Field --> tag for the data
class Product(BaseModel): #BaseModel --> Pydantic Model
    name :str = Field(description = "Name of the Product")
    price : str = Field(description="Price of the Product")
    description : str = Field(description="Description of the Product")
    category : str = Field(description="Category of the product")


In [44]:
qwen_model = ChatOllama(
    model = 'qwen2.5:1.5b',
    temperature=0
).with_structured_output(Product)
gemini_model = ChatGoogleGenerativeAI(
    model = 'gemini-3.8-flash',
    api_key = os.environ['Gemini_api'],
    temperature = 0
).with_structured_output(Product)

In [45]:
# Create --> LCEL
chain = prompt_template | gemini_model

In [46]:
user_input = [
    {
        "product" : data,
        "question"  : "What is the price of the Iphone 15 Pro Max?"
    },
    {
        "product" :data,
        "question" : "What is the price of the galaxy S24 Ultra?"
    },
    {
        "product" : data,
        "question" : "What is the Price of Redmi?"
    },
    {
        "product" : data,
        "question" : "What is the price of the OnePlus?"
    }
]

In [47]:
user_input = (
    {
        "product" : data,
        "question"  : "What is the price of the White T-shirt?"
    })

In [48]:
response = chain.invoke(user_input) # model_dump_json()--> Typecaste the output to json format

GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
response.model_dump_json()

In [ ]:
user_input = [
    {
        "product" : data,
        "question"  : "What is the price of the Iphone 15 Pro Max?"
    },
    {
        "product" :data,
        "question" : "What is the price of the galaxy S24 Ultra?"
    },
    {
        "product" : data,
        "question" : "What is the Price of Redmi?"
    },
    {
        "product" : data,
        "question" : "What is the price of the OnePlus?"
    }
]

In [ ]:
batch_response = chain.batch(user_input) 
# batch() --> Accepts multiple inputs
# invoke() --> For one input

In [ ]:
results = []
for result in batch_response:
    results.append(result.model_dump_json())
results

In [ ]:
results = []
for query in user_input:
    print(chain.invoke(query).model.dump())